# Initial Data Upload to AWS S3

This notebook performs a **one-time upload** of the initial datasets (`Resume.csv` and `job_title_des.csv`) to an AWS S3 bucket. This serves as the foundational raw data source for the entire resume matching MLOps pipeline.

**Pre-requisites:**
1.  Ensure you have `Resume.csv` and `job_title_des.csv` in the same directory as this notebook.
2.  Set up your AWS credentials in a `.env` file (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`).

In [5]:
import boto3
import os
from dotenv import load_dotenv
import logging

In [6]:
# --- Configuration ---
load_dotenv() # Load environment variables from .env file

# **ACTION REQUIRED**: Set your S3 bucket name here.
# It's recommended to create a new, dedicated bucket for this project.
BUCKET_NAME = "resume-matcher-bucket-sahil" 

# Local and S3 file paths
FILES_TO_UPLOAD = {
    "Resume.csv": "raw-data/Resume.csv",
    "job_title_des.csv": "raw-data/job_title_des.csv"
}

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [7]:
# --- Initialize S3 Client ---
try:
    s3_client = boto3.client(
        's3',
        aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
        aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
        region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1') # Default to us-east-1 if not set
    )
    logging.info("Successfully initialized S3 client.")
except Exception as e:
    logging.error(f"Failed to initialize S3 client: {e}")
    s3_client = None

2025-11-01 22:39:44,445 - INFO - Successfully initialized S3 client.


In [8]:
# --- Main Upload Logic ---
def upload_files_to_s3(client, bucket, files_map):
    """Iterates through a dictionary of files and uploads them to S3."""
    if not client:
        logging.error("S3 client is not available. Aborting upload.")
        return

    for local_path, s3_key in files_map.items():
        # 1. Check if the local file exists before trying to upload
        if not os.path.exists(local_path):
            logging.warning(f"Local file not found: '{local_path}'. Skipping upload.")
            continue
        
        # 2. Upload the file
        try:
            logging.info(f"Uploading '{local_path}' to s3://{bucket}/{s3_key}...")
            client.upload_file(local_path, bucket, s3_key)
            logging.info(f"✅ Successfully uploaded '{local_path}'.")
        except boto3.exceptions.S3UploadFailedError as e:
            logging.error(f"❌ Failed to upload '{local_path}'. Check your permissions and bucket name.")
            logging.error(f"   Details: {e}")
        except Exception as e:
            logging.error(f"❌ An unexpected error occurred while uploading '{local_path}': {e}")

if __name__ == "__main__":
    upload_files_to_s3(s3_client, BUCKET_NAME, FILES_TO_UPLOAD)
    logging.info("\n--- Data upload process finished. ---")


2025-11-01 22:39:44,455 - INFO - Uploading 'Resume.csv' to s3://resume-matcher-bucket-sahil/raw-data/Resume.csv...
2025-11-01 22:40:17,721 - INFO - ✅ Successfully uploaded 'Resume.csv'.
2025-11-01 22:40:17,721 - INFO - Uploading 'job_title_des.csv' to s3://resume-matcher-bucket-sahil/raw-data/job_title_des.csv...
2025-11-01 22:40:20,695 - INFO - ✅ Successfully uploaded 'job_title_des.csv'.
2025-11-01 22:40:20,695 - INFO - 
--- Data upload process finished. ---
